In [ ]:
# rs-fMRI viewer

import math
import os
import csv
from datetime import datetime
from functools import lru_cache
from tempfile import NamedTemporaryFile

import nibabel as nib
import numpy as np
import plotly.express as px

import pydicom
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import SecondaryCaptureImageStorage, ExplicitVRLittleEndian, generate_uid
from scipy.ndimage import label
from dash import Dash, Input, Output, State, dcc, html
from dash.exceptions import PreventUpdate
from skimage.transform import resize

# =========================
# CONFIG
# =========================
BASE_DIR = "/Users/forrestlin/Downloads/example_output_mni"

AXIS_LABELS = {
    0: "Sagittal",
    1: "Coronal",
    2: "Axial",
}

NETWORK_GRADIENTS = [
    (np.array([1.00, 0.00, 0.00]), np.array([0.00, 0.00, 1.00])),  # red → blue
    (np.array([0.00, 1.00, 0.00]), np.array([1.00, 0.00, 1.00])),  # green → magenta
    (np.array([1.00, 0.50, 0.00]), np.array([0.00, 0.00, 1.00])),  # orange → blue
    (np.array([1.00, 1.00, 0.00]), np.array([0.50, 0.00, 1.00])),  # yellow → violet
    (np.array([0.00, 1.00, 1.00]), np.array([1.00, 0.00, 0.00])),  # cyan → red
    (np.array([1.00, 0.00, 0.50]), np.array([0.00, 1.00, 0.50])),  # hot-pink → mint
    (np.array([0.00, 0.00, 1.00]), np.array([1.00, 0.80, 0.00])),  # blue → gold
    (np.array([1.00, 0.00, 1.00]), np.array([0.00, 1.00, 0.00])),  # magenta → green
    (np.array([0.00, 0.80, 1.00]), np.array([1.00, 0.30, 0.00])),  # sky-blue → burnt-orange
    (np.array([0.60, 0.00, 1.00]), np.array([0.00, 1.00, 0.40])),  # purple → lime
    (np.array([1.00, 0.20, 0.00]), np.array([0.00, 0.80, 1.00])),  # scarlet → azure
    (np.array([0.00, 1.00, 0.60]), np.array([1.00, 0.00, 0.40])),  # spring-green → rose-red
]

GRADIENT_PRESETS = {
    "red-blue": (np.array([1, 0, 0]), np.array([0, 0, 1])),
    "green-magenta": (np.array([0, 1, 0]), np.array([1, 0, 1])),
    "orange-blue": (np.array([1, 0.5, 0]), np.array([0, 0, 1])),
    "yellow-violet": (np.array([1, 1, 0]), np.array([0.5, 0, 1])),
    "cyan-red": (np.array([0, 1, 1]), np.array([1, 0, 0])),
    "blue-gold": (np.array([0, 0, 1]), np.array([1, 0.8, 0])),
}

SUFFIX_DISPLAY_NAMES = {
    "desc-brain_mask_t1":   "Brain Mask",
    "reho_t1":              "ReHo",
    "aud_net_t1":           "Auditory Network",
    "ec_net_t1":            "EC Network",
    "lfp_net_t1":           "Left Frontoparietal Network",
    "rfp_net_t1":           "Right Frontoparietal Network",
    "vl_net_t1":            "Visuolateral Network",
    "vm_net_t1":            "Visuomedial Network",
    "vo_net_t1":            "Visuooccipital Network",
    "dmn_net_t1":           "Default Mode Network",
    "seedcorr_t1":          "Seed Correlation (T1w)",
    "sm_net_t1":            "Sensorimotor Network",
}

ALL_SUFFIXES = list(SUFFIX_DISPLAY_NAMES.keys())

SUFFIX_GRADIENT_INDEX = {
    "aud_net_t1":           0,
    "desc-brain_mask_t1":   1,
    "dmn_net_t1":           2,
    "ec_net_t1":            3,
    "lfp_net_t1":           4,
    "reho_t1":              5,
    "rfp_net_t1":           6,
    "seedcorr_t1":          7,
    "vl_net_t1":            8,
    "vm_net_t1":            9,
    "vo_net_t1":            10,
    "sm_net_t1":            11,
}

CARD_STYLE = {
    "backgroundColor": "#f4f4f4",
    "borderRadius": "16px",
    "padding": "20px",
    "boxShadow": "0 1px 6px rgba(0,0,0,0.08)",
}

SECTION_TITLE_STYLE = {
    "fontSize": "18px",
    "fontWeight": "700",
    "marginBottom": "14px",
}

LABEL_STYLE = {
    "fontSize": "14px",
    "fontWeight": "600",
    "marginBottom": "8px",
    "display": "block",
}

DEFAULT_OPACITY = 0.70
DEFAULT_THRESHOLD = 0.50
DEFAULT_CLUSTER_SIZE = 20

# =========================
# DATA DISCOVERY
# =========================

# this function returns all folders in example_output_mni/ except the template folder
# this becomes the patient IDs used throughout the interface
def list_patients():
    if not os.path.isdir(BASE_DIR):
        return []

    patients = []

    for d in os.listdir(BASE_DIR):
        full_path = os.path.join(BASE_DIR, d)

        if not os.path.isdir(full_path):
            continue

        # skip template folders
        if d.lower() in ["template", "templates"]:
            continue

        patients.append(d)

    return sorted(patients)


PATIENTS = list_patients()
_fname_to_grad: dict[str, int] = {}

# subject/template specific display directory; obtains file path for either subject or template space
def patient_display_dir(patient_id, display_space="subject"):
    if display_space == "mni":
        return os.path.join(BASE_DIR, patient_id, "xprx", "mni")
    else:
        return os.path.join(BASE_DIR, patient_id, "xprx", "display")

# for each patient, it generates the patient-specific path to look for nifti files
def patient_root_dir(patient_id):
    return os.path.join(BASE_DIR, patient_id)

# depending on the selected space, it looks for all nifti files
# it is more restrictive for the template mni space; uses a preloaded prefix to filter out non-network files
def get_display_niftis(patient_id, display_space="subject"):
    if display_space == "mni":
        display_path = os.path.join(BASE_DIR, patient_id, "xprx", "mni")
        prefix = f"sub-{patient_id}_ses-1_task-rest_space-MNI152NLin2009cAsym_res-2_desc-"
    else:
        display_path = os.path.join(BASE_DIR, patient_id, "xprx", "display")
        prefix = f"sub-{patient_id}_ses-1_task-rest_acq-multiecho_"

    if not os.path.isdir(display_path):
        return {}

    if display_space == "mni":
        # MNI folder has non-network files too — filter strictly by prefix
        nii_files = [
            f for f in os.listdir(display_path)
            if (f.endswith(".nii") or f.endswith(".nii.gz")) and f.startswith(prefix)
        ]
    else:
        # Subject display/ folder only contains network maps — no prefix filter needed
        nii_files = [
            f for f in os.listdir(display_path)
            if f.endswith(".nii") or f.endswith(".nii.gz")
        ]

    file_dict = {}

    for f in sorted(nii_files):
        name = f.replace(".nii.gz", "").replace(".nii", "")
        # Strip prefix to get the bare suffix (e.g. "aud_net_t1")
        suffix = name.replace(prefix, "") if name.startswith(prefix) else name

        # Direct lookup — no rsplit normalization needed, suffixes are exact keys
        display_label = SUFFIX_DISPLAY_NAMES.get(suffix, suffix)
        gradient_idx = SUFFIX_GRADIENT_INDEX.get(suffix, 0)
        file_dict[display_label] = (f, gradient_idx)
        _fname_to_grad[f] = gradient_idx

    return file_dict


for _p in PATIENTS:
    get_display_niftis(_p)

# searches for the anatomical T1 scan by searching through the entire patient directory using preferred terms
# looks for files that have "T1w", "T1", "t1w", "t1", "desc-preproc_T1w" in the filename
def find_t1_file(patient_id):
    root = patient_root_dir(patient_id)
    if not os.path.isdir(root):
        return None

    # Absolute path of the MNI dir to exclude from the walk
    mni_dir = os.path.abspath(os.path.join(root, "xprx", "mni"))

    preferred_terms = ["desc-preproc_T1w", "T1w", "T1", "t1w", "t1"]
    # Also exclude any file that looks like an MNI template even if found elsewhere
    exclude_terms = ["tpl-", "space-MNI", "MNI152"]
    candidates = []

    for dirpath, dirnames, filenames in os.walk(root):
        # Prevent descending into xprx/mni/
        dirnames[:] = [
            d for d in dirnames
            if not os.path.abspath(os.path.join(dirpath, d)).startswith(mni_dir)
        ]

        for fname in filenames:
            if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
                continue
            # Skip MNI template files by name
            if any(term in fname for term in exclude_terms):
                continue
            full_path = os.path.join(dirpath, fname)
            score = 0
            for i, term in enumerate(preferred_terms):
                if term in fname:
                    score = max(score, len(preferred_terms) - i)
            if score > 0:
                candidates.append((score, full_path))

    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][1]


# =========================
# VOLUME HELPERS
# =========================
@lru_cache(maxsize=256)
def load_nifti(path):
    img = nib.load(path)
    vol = img.get_fdata().astype(np.float32)
    if vol.ndim == 4:
        vol = vol[..., 0]
    return np.nan_to_num(vol)


@lru_cache(maxsize=256)
def load_patient_overlay(patient_id, filename, display_space="subject"):
    if display_space == "mni":
        path = os.path.join(BASE_DIR, patient_id, "xprx", "mni", filename)
        return load_nifti(path)

    # Subject space: resample overlay into T1 voxel grid using affines
    overlay_path = os.path.join(BASE_DIR, patient_id, "xprx", "display", filename)
    overlay_img = nib.load(overlay_path)
    overlay_data = overlay_img.get_fdata().astype(np.float32)
    if overlay_data.ndim == 4:
        overlay_data = overlay_data[..., 0]
    overlay_data = np.nan_to_num(overlay_data)

    t1_path = find_t1_file(patient_id)
    if t1_path is None:
        return overlay_data  # no T1 to align to, return as-is

    t1_img = nib.load(t1_path)
    t1_shape = t1_img.shape[:3]

    if overlay_data.shape == tuple(t1_shape):
        return overlay_data  # already aligned, skip resampling

    # Map from T1 voxel coords → world → overlay voxel coords
    t1_to_overlay = np.linalg.inv(overlay_img.affine) @ t1_img.affine

    from scipy.ndimage import affine_transform
    resampled = affine_transform(
        overlay_data,
        t1_to_overlay[:3, :3],
        offset=t1_to_overlay[:3, 3],
        output_shape=tuple(t1_shape),
        order=1,
        mode='constant',
        cval=0.0,
    )
    return resampled.astype(np.float32)


@lru_cache(maxsize=64)
def load_patient_t1(patient_id):
    t1_path = find_t1_file(patient_id)
    if t1_path is None:
        return None
    return load_nifti(t1_path)

@lru_cache(maxsize=64)
def load_patient_mni_template(patient_id):
    mni_path = os.path.join(
        BASE_DIR,
        patient_id,
        "xprx",
        "mni",
        "tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz"
    )
    if not os.path.exists(mni_path):
        return None
    return load_nifti(mni_path)

def normalize_to_unit(arr):
    arr = np.asarray(arr, dtype=np.float32)
    vmin, vmax = float(np.min(arr)), float(np.max(arr))
    if vmax <= vmin:
        return np.zeros_like(arr)
    return (arr - vmin) / (vmax - vmin)

def get_base_volume(patient_id, display_space, selected_files):
    if display_space == "mni":
        base_vol = load_patient_mni_template(patient_id)
        if base_vol is None:
            # fallback: use first overlay
            if selected_files:
                base_vol = load_patient_overlay(patient_id, selected_files[0], display_space)
            else:
                return None
    else:
        base_vol = load_patient_t1(patient_id)

    if base_vol is None:
        if not selected_files:
            return None
        base_vol = load_patient_overlay(patient_id, selected_files[0], display_space)

    return base_vol

def clip_percentile(sl, p_low=1, p_high=99):
    lo, hi = np.percentile(sl, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(sl, dtype=np.float32)
    return np.clip(sl, lo, hi).astype(np.float32)


def get_slice(vol3d, axis, idx):
    if axis == 0:
        sl = vol3d[idx, :, :]
    elif axis == 1:
        sl = vol3d[:, idx, :]
    else:
        sl = vol3d[:, :, idx]
    return np.rot90(sl)


def gradient_color(t, low_color, high_color):
    t = np.asarray(t, dtype=np.float32)[..., np.newaxis]
    return (1.0 - t) * low_color + t * high_color


def _css_gradient(low_color, high_color):
    lo = [int(x) for x in (255 * np.clip(low_color, 0, 1))]
    hi = [int(x) for x in (255 * np.clip(high_color, 0, 1))]

    return (
        f"linear-gradient(to right, "
        f"rgb({lo[0]},{lo[1]},{lo[2]}), "
        f"rgb({hi[0]},{hi[1]},{hi[2]}))"
    )


def hex_to_rgb_array(hex_color):
    hex_color = hex_color.lstrip("#")
    return np.array([int(hex_color[i:i+2], 16) for i in (0, 2, 4)], dtype=np.float32) / 255.0


def apply_cluster_threshold(mask, min_cluster_size=20, connectivity=1):
    if not np.any(mask):
        return mask

    structure = None
    if mask.ndim == 2:
        structure = np.ones((3, 3)) if connectivity == 2 else None
    elif mask.ndim == 3:
        structure = np.ones((3, 3, 3)) if connectivity == 2 else None

    labeled, num_features = label(mask, structure=structure)
    if num_features == 0:
        return mask

    counts = np.bincount(labeled.ravel())
    valid_labels = np.where(counts >= min_cluster_size)[0]
    valid_labels = valid_labels[valid_labels != 0]
    filtered_mask = np.isin(labeled, valid_labels)
    return filtered_mask

def get_network_suffix(fname):
    if not fname:
        return None

    base = os.path.basename(fname)

    # subject-space names
    for sfx in SUFFIX_DISPLAY_NAMES.keys():
        if sfx in base:
            return sfx

    # MNI names
    mapping = {
        "desc-aud_": "aud_net_t1",
        "desc-ec_": "ec_net_t1",
        "desc-lfp_": "lfp_net_t1",
        "desc-rfp_": "rfp_net_t1",
        "desc-vl_": "vl_net_t1",
        "desc-vm_": "vm_net_t1",
        "desc-vo_": "vo_net_t1",
        "desc-dmn_": "dmn_net_t1",
        "desc-sm_": "sm_net_t1",
        "desc-reho_": "reho_t1",
        "desc-seedcorr_": "seedcorr_t1",
    }

    for pattern, suffix in mapping.items():
        if pattern in base:
            return suffix

    print(f"WARNING: no suffix found for {fname}")
    return None

def get_overlay_colors(fname, gradient_settings):
    suffix = get_network_suffix(fname)
    entry = (gradient_settings or {}).get(suffix, {})

    if "low" in entry and "high" in entry:
        low_c = np.array(entry["low"], dtype=np.float32)
        high_c = np.array(entry["high"], dtype=np.float32)
        return low_c, high_c


    g_idx = SUFFIX_GRADIENT_INDEX.get(suffix, 0)

    return NETWORK_GRADIENTS[g_idx]


def blend_single_slice(base_vol, overlay_vols, axis, idx, per_overlay_settings,
                       gradient_settings, selected_files, display_space="subject"):
    base_slice = normalize_to_unit(clip_percentile(get_slice(base_vol, axis, idx), 1, 99))
    rgb = np.stack([base_slice, base_slice, base_slice], axis=-1)
    target_shape = base_slice.shape

    for i, vol in enumerate(overlay_vols):
        try:
            settings = per_overlay_settings[i]
            threshold = float(settings.get("threshold", DEFAULT_THRESHOLD))
            alpha = float(np.clip(settings.get("opacity", DEFAULT_OPACITY), 0, 1))
            min_cluster_size = int(settings.get("cluster_size", DEFAULT_CLUSTER_SIZE))

            overlay_slice = normalize_to_unit(get_slice(vol, axis, idx))

            # MNI only: template and overlay may still have slightly different dims
            if display_space == "mni" and overlay_slice.shape != target_shape:
                overlay_slice = resize(
                    overlay_slice, target_shape,
                    order=1, preserve_range=True, anti_aliasing=False
                ).astype(np.float32)

            mask = overlay_slice >= threshold
            mask = apply_cluster_threshold(mask, min_cluster_size=min_cluster_size)
            if not np.any(mask):
                continue

            fname = selected_files[i]
            low_c, high_c = get_overlay_colors(fname, gradient_settings)

            t_vals = np.zeros_like(overlay_slice, dtype=np.float32)
            above = overlay_slice[mask]
            if above.size > 0 and above.max() > threshold:
                t_vals[mask] = (above - threshold) / (1.0 - threshold + 1e-8)

            print("MASK VOXELS:", np.sum(mask))
            print("T RANGE:", t_vals.min(), t_vals.max())
            grad_colors = gradient_color(t_vals, low_c, high_c)
            print(
                "GRAD SAMPLE:",
                grad_colors[mask][:5] if np.any(mask) else "empty"
            )
            mask_3d = np.stack([mask] * 3, axis=-1)
            rgb[mask_3d] = (1.0 - alpha) * rgb[mask_3d] + alpha * grad_colors[mask_3d]

        except Exception as e:
            print(f"[blend_single_slice] ERROR on overlay {i} ({selected_files[i]}): {e}")
            import traceback; traceback.print_exc()
            continue

    return np.clip(rgb, 0, 1)

def empty_figure(message="No image available"):
    fig = px.imshow(np.zeros((16, 16)))
    fig.update_layout(
        template="plotly_white",
        xaxis={"visible": False},
        yaxis={"visible": False},
        coloraxis_showscale=False,
        annotations=[
            dict(
                text=message,
                x=0.5,
                y=0.5,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(size=18, color="#666"),
            )
        ],
        margin=dict(l=10, r=10, t=10, b=10),
    )
    return fig


def make_mosaic_rgb(base_vol, overlay_vols, axis, start, step, n_slices, pad, per_overlay_settings, gradient_settings, selected_files, display_space="subject"):
    max_idx = base_vol.shape[axis] - 1
    start = int(np.clip(start, 0, max_idx))
    step = max(1, int(step))
    n_slices = max(1, int(n_slices))

    indices = list(range(start, max_idx + 1, step))[:n_slices]
    if not indices:
        indices = [start]

    slices = [
        blend_single_slice(
            base_vol=base_vol,
            overlay_vols=overlay_vols,
            axis=axis,
            idx=i,
            per_overlay_settings=per_overlay_settings,
            gradient_settings=gradient_settings,
            selected_files=selected_files,
            display_space=display_space,
        )
        for i in indices
    ]

    cols = int(math.ceil(math.sqrt(len(slices))))
    rows = int(math.ceil(len(slices) / cols))
    h, w, _ = slices[0].shape

    mosaic = np.zeros(
        (rows * h + (rows - 1) * pad, cols * w + (cols - 1) * pad, 3),
        dtype=np.float32,
    )

    for k, sl in enumerate(slices):
        r, c = divmod(k, cols)
        y0 = r * (h + pad)
        x0 = c * (w + pad)
        mosaic[y0:y0 + h, x0:x0 + w, :] = sl

    return mosaic, indices


def render_current_rgb(patient_id, selected_files, settings, mode, axis, idx, mosaic_n, gradient_settings, display_space):
    if not patient_id:
        raise ValueError("No patient selected")

    settings = settings or {}
    gradient_settings = gradient_settings or {}
    selected_files = selected_files or []
    axis = int(axis)
    idx = int(idx)
    mosaic_n = int(mosaic_n)

    base_vol = get_base_volume(patient_id, display_space, selected_files)
    if base_vol is None:
        if not selected_files:
            raise ValueError("No T1 file found and no overlay selected")
        base_vol = load_patient_overlay(patient_id, selected_files[0], display_space)

    idx = max(0, min(idx, base_vol.shape[axis] - 1))
    overlay_vols = [load_patient_overlay(patient_id, f, display_space) for f in selected_files]
    per_overlay_settings = [
        settings.get(
            f,
            {
                "opacity": DEFAULT_OPACITY,
                "threshold": DEFAULT_THRESHOLD,
                "cluster_size": DEFAULT_CLUSTER_SIZE,
            },
        )
        for f in selected_files
    ]

    if mode == "single":
        rgb = blend_single_slice(
            base_vol=base_vol,
            overlay_vols=overlay_vols,
            axis=axis,
            idx=idx,
            per_overlay_settings=per_overlay_settings,
            gradient_settings=gradient_settings,
            selected_files=selected_files,
            display_space=display_space,
        )
        title = f"{patient_id}_{AXIS_LABELS[axis].lower()}_slice_{idx}"
        return (np.clip(rgb, 0, 1) * 255).astype(np.uint8), title

    mosaic_rgb, indices = make_mosaic_rgb(
        base_vol=base_vol,
        overlay_vols=overlay_vols,
        axis=axis,
        start=idx,
        step=1,
        n_slices=mosaic_n,
        pad=2,
        per_overlay_settings=per_overlay_settings,
        gradient_settings=gradient_settings,
        selected_files=selected_files,
        display_space=display_space
    )
    title = f"{patient_id}_{AXIS_LABELS[axis].lower()}_mosaic_start_{idx}_n_{len(indices)}"
    return (np.clip(mosaic_rgb, 0, 1) * 255).astype(np.uint8), title

def render_full_rgb_volume(
    patient_id,
    selected_files,
    settings,
    gradient_settings,
    display_space,
):
    base_vol = get_base_volume(
        patient_id,
        display_space,
        selected_files
    )

    overlay_vols = [
        load_patient_overlay(patient_id, f, display_space)
        for f in selected_files
    ]

    per_overlay_settings = [
        settings.get(
            f,
            {
                "opacity": DEFAULT_OPACITY,
                "threshold": DEFAULT_THRESHOLD,
                "cluster_size": DEFAULT_CLUSTER_SIZE,
            },
        )
        for f in selected_files
    ]

    n_slices = base_vol.shape[2]

    rgb_volume = []

    for z in range(n_slices):

        rgb = blend_single_slice(
            base_vol=base_vol,
            overlay_vols=overlay_vols,
            axis=2,
            idx=z,
            per_overlay_settings=per_overlay_settings,
            gradient_settings=gradient_settings,
            selected_files=selected_files,
            display_space=display_space,
        )

        rgb_volume.append(
            (np.clip(rgb, 0, 1) * 255).astype(np.uint8)
        )

    rgb_volume = np.stack(rgb_volume, axis=0)

    return rgb_volume

def save_rgb_volume_as_dicom_series(
    rgb_volume,
    output_dir,
    patient_id,
):
    os.makedirs(output_dir, exist_ok=True)

    study_uid = generate_uid()
    series_uid = generate_uid()

    for i in range(rgb_volume.shape[0]):

        rgb = rgb_volume[i]

        filename = os.path.join(
            output_dir,
            f"slice_{i:04d}.dcm"
        )

        file_meta = FileMetaDataset()
        file_meta.MediaStorageSOPClassUID = SecondaryCaptureImageStorage
        file_meta.MediaStorageSOPInstanceUID = generate_uid()
        file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

        ds = FileDataset(
            filename,
            {},
            file_meta=file_meta,
            preamble=b"\0" * 128
        )

        ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
        ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID

        ds.PatientID = str(patient_id)
        ds.PatientName = str(patient_id)

        ds.StudyInstanceUID = study_uid
        ds.SeriesInstanceUID = series_uid

        ds.Modality = "OT"

        ds.Rows = rgb.shape[0]
        ds.Columns = rgb.shape[1]

        ds.SamplesPerPixel = 3
        ds.PhotometricInterpretation = "RGB"
        ds.PlanarConfiguration = 0

        ds.BitsAllocated = 8
        ds.BitsStored = 8
        ds.HighBit = 7
        ds.PixelRepresentation = 0

        ds.InstanceNumber = i + 1

        ds.PixelData = rgb.tobytes()

        ds.save_as(filename, write_like_original=False)

def save_rgb_secondary_capture(
    rgb_u8,
    out_path,
    patient_id="anon",
    series_description="Dash Overlay Export"
):
    if rgb_u8.ndim != 3 or rgb_u8.shape[2] != 3:
        raise ValueError("Expected RGB image with shape (rows, cols, 3)")

    rgb_u8 = np.ascontiguousarray(rgb_u8, dtype=np.uint8)
    rows, cols, _ = rgb_u8.shape
    now = datetime.now()

    file_meta = FileMetaDataset()
    file_meta.FileMetaInformationVersion = b"\x00\x01"
    file_meta.MediaStorageSOPClassUID = SecondaryCaptureImageStorage
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    file_meta.ImplementationClassUID = generate_uid()

    ds = FileDataset(str(out_path), {}, file_meta=file_meta, preamble=b"\0" * 128)

    # Required identity tags
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.StudyInstanceUID = generate_uid()
    ds.SeriesInstanceUID = generate_uid()
    ds.FrameOfReferenceUID = generate_uid()

    # Patient / study / series
    ds.PatientID = str(patient_id)
    ds.PatientName = str(patient_id)
    ds.Modality = "OT"
    ds.ConversionType = "WSD"  # workstation
    ds.ImageType = ["DERIVED", "SECONDARY", "OTHER"]

    ds.StudyDate = now.strftime("%Y%m%d")
    ds.StudyTime = now.strftime("%H%M%S")
    ds.SeriesDate = now.strftime("%Y%m%d")
    ds.SeriesTime = now.strftime("%H%M%S")
    ds.ContentDate = now.strftime("%Y%m%d")
    ds.ContentTime = now.strftime("%H%M%S")

    ds.StudyID = "1"
    ds.SeriesNumber = 1
    ds.InstanceNumber = 1
    ds.SeriesDescription = series_description

    # RGB image tags
    ds.Rows = rows
    ds.Columns = cols
    ds.SamplesPerPixel = 3
    ds.PhotometricInterpretation = "RGB"
    ds.PlanarConfiguration = 0   # RGBRGBRGB...
    ds.BitsAllocated = 8
    ds.BitsStored = 8
    ds.HighBit = 7
    ds.PixelRepresentation = 0

    # Helpful extras
    ds.PixelAspectRatio = [1, 1]

    ds.PixelData = rgb_u8.tobytes()

    ds.is_little_endian = True
    ds.is_implicit_VR = False

    ds.save_as(str(out_path), write_like_original=False)


# =========================
# 4) APP
# =========================
app = Dash(__name__)
app.title = "rs-fMRI Clinical Interface"

default_patient = PATIENTS[0] if PATIENTS else None

app.layout = html.Div(
    style={
        "backgroundColor": "#ececec",
        "minHeight": "100vh",
        "padding": "16px",
        "fontFamily": "Inter, system-ui, -apple-system, Segoe UI, Roboto, sans-serif",
    },
    children=[
        dcc.Store(id="overlay_settings", data={}),
        dcc.Store(id="network_ratings", data={}),
        dcc.Store(id="gradient_settings", data={}),
        dcc.Store(id="overall_patient_ratings", data={}),

        html.Div(
            style={
                **CARD_STYLE,
                "marginBottom": "16px",
                "padding": "22px 28px",
                "display": "flex",
                "justifyContent": "center",
                "alignItems": "center",
            },
            children=[
                html.H1(
                    "rs-fMRI Clinical Interface",
                    style={"margin": 0, "fontSize": "42px", "fontWeight": "800", "letterSpacing": "0.5px"},
                )
            ],
        ),

        html.Div(
            style={
                "display": "grid",
                "gridTemplateColumns": "300px minmax(700px, 1fr) 340px",
                "gap": "16px",
                "alignItems": "start",
            },
            children=[
                # LEFT
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Patient Selection", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("Patient", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="patient_dropdown",
                                    options=[{"label": p, "value": p} for p in PATIENTS],
                                    value=default_patient,
                                    clearable=False,
                                ),
                                html.Div(style={"height": "16px"}),
                                html.Label("Active Overlays", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="nifti_dropdown",
                                    multi=True,
                                    placeholder="Select overlays to display",
                                ),
                            ],
                        ),
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Viewing Controls", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("Display Space", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="display_space",
                                    options=[
                                        {"label": "Subject (T1)", "value": "subject"},
                                        {"label": "Template (MNI)", "value": "mni"},
                                    ],
                                    value="subject",
                                    labelStyle={"display": "block", "marginBottom": "8px"},
                                    inputStyle={"marginRight": "8px"},
                                ),
                                html.Label("View Mode", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="mode",
                                    options=[
                                        {"label": "Single slice", "value": "single"},
                                        {"label": "Mosaic", "value": "mosaic"},
                                    ],
                                    value="single",
                                    labelStyle={"display": "block", "marginBottom": "8px"},
                                    inputStyle={"marginRight": "8px"},
                                ),
                                html.Div(style={"height": "18px"}),
                                html.Label("Plane", style=LABEL_STYLE),
                                dcc.RadioItems(
                                    id="axis",
                                    options=[
                                        {"label": "Sagittal", "value": 0},
                                        {"label": "Coronal", "value": 1},
                                        {"label": "Axial", "value": 2},
                                    ],
                                    value=2,
                                    inline=True,
                                    labelStyle={"marginRight": "16px"},
                                    inputStyle={"marginRight": "6px"},
                                ),
                                html.Div(style={"height": "18px"}),

                                html.Label("Overall Patient Rating", style=LABEL_STYLE),

                                dcc.Dropdown(
                                    id="overall_patient_rating_dropdown",
                                    options=[
                                        {"label": "0", "value": 0},
                                        {"label": "1", "value": 1},
                                        {"label": "2", "value": 2},
                                        {"label": "3", "value": 3},
                                    ],
                                    placeholder="Select overall rating",
                                    clearable=True,
                                ),

                                html.Div(style={"height": "18px"}),

                                html.Button(
                                    "Save Ratings to CSV",
                                    id="save_ratings_button",
                                    n_clicks=0,
                                    style={
                                        "backgroundColor": "#4CAF50",
                                        "color": "white",
                                        "border": "none",
                                        "padding": "10px 16px",
                                        "borderRadius": "8px",
                                        "cursor": "pointer",
                                        "fontWeight": "600",
                                        "marginRight": "10px",
                                    },
                                ),
                                html.Button(
                                    "Export Current View as DICOM",
                                    id="export_dicom_button",
                                    n_clicks=0,
                                    style={
                                        "backgroundColor": "#2563eb",
                                        "color": "white",
                                        "border": "none",
                                        "padding": "10px 16px",
                                        "borderRadius": "8px",
                                        "cursor": "pointer",
                                        "fontWeight": "600",
                                        "marginTop": "12px",
                                    },
                                ),
                                dcc.Download(id="download_dicom"),
                                html.Button(
                                    "Export Full Volume",
                                    id="export_volume_button",
                                    n_clicks=0,
                                    style={
                                        "backgroundColor": "#7c3aed",
                                        "color": "white",
                                        "border": "none",
                                        "padding": "10px 16px",
                                        "borderRadius": "8px",
                                        "cursor": "pointer",
                                        "fontWeight": "600",
                                        "marginTop": "12px",
                                    },
                                ),
                                dcc.Download(id="download_volume"),
                            ],
                        ),
                    ],
                ),

                # CENTER
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div(
                                    style={
                                        "display": "flex",
                                        "justifyContent": "space-between",
                                        "alignItems": "center",
                                        "marginBottom": "10px",
                                    },
                                    children=[
                                        html.Div("Image Viewer", style={**SECTION_TITLE_STYLE, "marginBottom": 0}),
                                        html.Div(id="note", style={"fontSize": "13px", "color": "#666", "textAlign": "right"}),
                                    ],
                                ),
                                dcc.Graph(
                                    id="view",
                                    style={"height": "760px"},
                                    config={"displayModeBar": False},
                                    figure=empty_figure("Select a patient to begin"),
                                ),
                                html.Div(style={"height": "6px"}),
                                html.Label("Slice", style=LABEL_STYLE),
                                dcc.Slider(id="idx", min=0, max=100, step=1, value=50, updatemode="drag"),
                                html.Div(style={"height": "20px"}),
                                html.Label("Mosaic slices", style=LABEL_STYLE),
                                dcc.Slider(
                                    id="mosaic_n",
                                    min=4,
                                    max=100,
                                    step=4,
                                    value=36,
                                    marks={4: "4", 16: "16", 36: "36", 64: "64", 100: "100"},
                                    updatemode="drag",
                                ),
                            ],
                        ),
                    ],
                ),

                # RIGHT
                html.Div(
                    style={"display": "flex", "flexDirection": "column", "gap": "16px"},
                    children=[
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Network Legend", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Div(id="network_legend"),
                            ],
                        ),
                        html.Div(
                            style=CARD_STYLE,
                            children=[
                                html.Div("Overlay Controls", style={**SECTION_TITLE_STYLE, "fontSize": "20px"}),
                                html.Label("Select Active Overlay", style=LABEL_STYLE),
                                dcc.Dropdown(
                                    id="active_overlay_dropdown",
                                    placeholder="Pick an overlay to adjust",
                                    clearable=True,
                                ),
                                html.Div(style={"height": "18px"}),
                                html.Div(id="active_overlay_controls"),
                            ],
                        ),
                    ],
                ),
            ],
        ),
    ],
)


# =========================
# 5) CALLBACKS
# =========================
@app.callback(
    Output("nifti_dropdown", "options"),
    Output("nifti_dropdown", "value"),
    Input("patient_dropdown", "value"),
    Input("display_space", "value"),
    State("nifti_dropdown", "value"),
)
def update_nifti_dropdown(patient_id, display_space, current_selection):
    if not patient_id:
        return [], []
    file_dict = get_display_niftis(patient_id, display_space)
    if not file_dict:
        return [], []
    options = [
        {"label": label, "value": fname}
        for label, (fname, _) in sorted(file_dict.items())
    ]

    available = {opt["value"] for opt in options}

    preserved = [
        f for f in (current_selection or [])
        if f in available
    ]

    return options, preserved


@app.callback(
    Output("active_overlay_dropdown", "options"),
    Output("active_overlay_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    State("active_overlay_dropdown", "value"),
)
def sync_active_overlay_options(selected_files, current_active):

    selected_files = selected_files or []

    options = []

    for fname in selected_files:
        suffix = get_network_suffix(fname)

        if suffix:
            label = SUFFIX_DISPLAY_NAMES.get(suffix, suffix)
        else:
            label = fname

        options.append({
            "label": label,
            "value": fname
        })

    new_active = (
        current_active
        if current_active in selected_files
        else (selected_files[0] if selected_files else None)
    )

    return options, new_active

@app.callback(
    Output("active_overlay_controls", "children"),
    Input("active_overlay_dropdown", "value"),
    State("nifti_dropdown", "options"),
    State("overlay_settings", "data"),
    State("network_ratings", "data"),
    State("gradient_settings", "data"),
    State("patient_dropdown", "value"),
)
def render_active_controls(active_fname, nifti_options, settings, ratings, gradient_settings, patient_id):
    if not active_fname:
        return html.Div("No overlay selected.", style={"color": "#666", "fontSize": "14px"})

    label_lookup = {opt["value"]: opt["label"] for opt in (nifti_options or [])}
    label_text = label_lookup.get(active_fname, active_fname)

    low_c, high_c = get_overlay_colors(active_fname, gradient_settings or {})
    grad_css = _css_gradient(low_c, high_c)

    def rgb_to_hex(rgb):
        return "#" + "".join(f"{int(np.clip(255 * x, 0, 255)):02x}" for x in rgb)

    low_hex = rgb_to_hex(low_c)
    high_hex = rgb_to_hex(high_c)

    saved = (settings or {}).get(active_fname, {})
    opacity_val = saved.get("opacity", DEFAULT_OPACITY)
    threshold_val = saved.get("threshold", DEFAULT_THRESHOLD)
    cluster_size_val = saved.get("cluster_size", DEFAULT_CLUSTER_SIZE)

    ratings = ratings or {}
    patient_ratings = ratings.get(patient_id, {})

    suffix = get_network_suffix(active_fname)

    current_rating = patient_ratings.get(suffix)

    return html.Div([
        html.Div(
            style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "16px"},
            children=[
                html.Div(
                    style={
                        "width": "48px",
                        "height": "14px",
                        "borderRadius": "3px",
                        "backgroundImage": grad_css,
                        "flexShrink": 0,
                    }
                ),
                html.Div(label_text, style={"fontWeight": "700", "fontSize": "15px"}),
            ],
        ),
        html.Label("Opacity", style=LABEL_STYLE),
        dcc.Slider(
            id="active_opacity_slider",
            min=0,
            max=1,
            step=0.05,
            value=opacity_val,
            marks={0: "0", 0.5: "0.5", 1: "1"},
            updatemode="drag",
            tooltip={"placement": "bottom", "always_visible": True},
        ),
        html.Div(style={"height": "18px"}),
        html.Label("Threshold", style=LABEL_STYLE),
        dcc.Slider(
            id="active_threshold_slider",
            min=0,
            max=1,
            step=0.01,
            value=threshold_val,
            marks={0: "0", 0.5: "0.5", 1: "1"},
            updatemode="drag",
            tooltip={"placement": "bottom", "always_visible": True},
        ),
        html.Label("Cluster Size", style=LABEL_STYLE),
        dcc.Slider(
            id="active_cluster_slider",
            min=0,
            max=200,
            step=5,
            value=cluster_size_val,
            marks={0: "0", 50: "50", 100: "100", 200: "200"},
            updatemode="drag",
        ),
        html.Div(
            id="active_threshold_text",
            style={"marginTop": "10px", "fontSize": "13px", "color": "#555"},
            children=f"{threshold_val:.2f} ({int(round(threshold_val * 100))}% of max)",
        ),
        html.Div(style={"height": "18px"}),

        html.Label("Network Rating", style=LABEL_STYLE),
        dcc.Dropdown(
            id="network_rating_dropdown",
            options=[
                {"label": "0", "value": 0},
                {"label": "1", "value": 1},
                {"label": "2", "value": 2},
                {"label": "3", "value": 3},
            ],
            placeholder="Select rating",
            value=current_rating,
            clearable=True,
        ),

        html.Div(style={"height": "20px"}),

        html.Label("Custom Gradient", style=LABEL_STYLE),
        html.Div([
            dcc.Input(
                id="low_color_picker",
                type="color",
                value=low_hex,
                style={"marginRight": "10px"},
            ),
            dcc.Input(
                id="high_color_picker",
                type="color",
                value=high_hex,
            ),
        ]),
    ])


@app.callback(
    Output("gradient_settings", "data"),
    Input("low_color_picker", "value"),
    Input("high_color_picker", "value"),
    State("active_overlay_dropdown", "value"),
    State("gradient_settings", "data"),
    prevent_initial_call=True,
)
def save_gradient_setting(low_hex, high_hex, active_fname, current):
    if not active_fname:
        raise PreventUpdate

    data = dict(current or {})
    suffix = get_network_suffix(active_fname)
    print("SAVING:", suffix)
    print("LOW HEX:", low_hex)
    print("HIGH HEX:", high_hex)
    entry = data.get(suffix, {})

    if low_hex and high_hex:
        entry["low"] = hex_to_rgb_array(low_hex).tolist()
        entry["high"] = hex_to_rgb_array(high_hex).tolist()

    data[suffix] = entry
    print("NEW GRADIENT SETTINGS:", data)
    return data


@app.callback(
    Output("active_threshold_text", "children"),
    Input("active_threshold_slider", "value"),
    prevent_initial_call=True,
)
def update_threshold_text(threshold):
    if threshold is None:
        raise PreventUpdate
    threshold = float(threshold)
    return f"{threshold:.2f} ({int(round(threshold * 100))}% of max)"


@app.callback(
    Output("network_ratings", "data"),
    Input("network_rating_dropdown", "value"),
    State("active_overlay_dropdown", "value"),
    State("patient_dropdown", "value"),
    State("network_ratings", "data"),
    prevent_initial_call=True,
)
def save_network_rating(rating, active_fname, patient_id, current_ratings):
    if active_fname is None or patient_id is None:
        raise PreventUpdate

    ratings = dict(current_ratings or {})
    if patient_id not in ratings:
        ratings[patient_id] = {}

    suffix = get_network_suffix(active_fname)

    if suffix:
        ratings[patient_id][suffix] = rating

    return ratings


@app.callback(
    Output("save_ratings_button", "children"),
    Input("save_ratings_button", "n_clicks"),
    State("network_ratings", "data"),
    State("overall_patient_ratings", "data"),
    prevent_initial_call=True,
)
def save_ratings_to_csv(n_clicks, ratings, overall_ratings):
    if not ratings:
        raise PreventUpdate
    
    overall_ratings = overall_ratings or {}

    filename = f"network_ratings_all_patients_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    filepath = os.path.join(BASE_DIR, filename)

    with open(filepath, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)

        header = (
            ["patient_id", "overall_rating"]
            + [
                SUFFIX_DISPLAY_NAMES.get(suffix, suffix)
                for suffix in ALL_SUFFIXES
            ]
        )
        writer.writerow(header)

        for patient_id in PATIENTS:
            patient_data = ratings.get(patient_id, {})
            overall_rating = overall_ratings.get(patient_id)
            row = [patient_id, overall_rating]
            for suffix in ALL_SUFFIXES:
                row.append(patient_data.get(suffix, ""))
            writer.writerow(row)

    return "Saved!"


@app.callback(
    Output("overlay_settings", "data"),
    Input("active_opacity_slider", "value"),
    Input("active_threshold_slider", "value"),
    Input("active_cluster_slider", "value"),
    State("active_overlay_dropdown", "value"),
    State("overlay_settings", "data"),
    prevent_initial_call=True,
)
def save_overlay_settings(opacity, threshold, cluster_size, active_fname, current_settings):
    if not active_fname:
        raise PreventUpdate

    settings = dict(current_settings or {})
    settings[active_fname] = {
        "opacity": float(opacity) if opacity is not None else DEFAULT_OPACITY,
        "threshold": float(threshold) if threshold is not None else DEFAULT_THRESHOLD,
        "cluster_size": int(cluster_size) if cluster_size is not None else DEFAULT_CLUSTER_SIZE,
    }
    return settings


@app.callback(
    Output("network_legend", "children"),
    Input("nifti_dropdown", "value"),
    Input("gradient_settings", "data"),
    State("nifti_dropdown", "options"),
)
def update_legend(selected_files, gradient_settings, options):
    selected_files = selected_files or []
    print("SELECTED FILES:", selected_files)
    gradient_settings = gradient_settings or {}

    if not selected_files:
        return html.Div("No overlays selected.", style={"color": "#666"})

    label_lookup = {opt["value"]: opt["label"] for opt in (options or [])}

    items = [
        html.Div(
            style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "10px"},
            children=[
                html.Div(
                    style={
                        "width": "48px",
                        "height": "14px",
                        "borderRadius": "3px",
                        "backgroundImage": "linear-gradient(to right, #888, #fff)",
                    }
                ),
                html.Div("T1 anatomical base", style={"fontSize": "14px", "fontWeight": "600"}),
            ],
        )
    ]

    for fname in selected_files:
        label_text = label_lookup.get(fname, fname)
        print("LEGEND FILE:", fname)
        print("SUFFIX:", get_network_suffix(fname))

        low_c, high_c = get_overlay_colors(fname, gradient_settings)

        print("LOW:", low_c)
        print("HIGH:", high_c)

        items.append(
            html.Div(
                style={"display": "flex", "alignItems": "center", "gap": "10px", "marginBottom": "10px"},
                children=[
                    html.Div(
                        style={
                            "width": "48px",
                            "height": "14px",
                            "borderRadius": "3px",
                            "backgroundImage": _css_gradient(low_c, high_c),
                        }
                    ),
                    html.Div(label_text, style={"fontSize": "14px", "fontWeight": "600"}),
                ],
            )
        )

    return items


@app.callback(
    Output("idx", "max"),
    Output("idx", "value"),
    Output("note", "children"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("axis", "value"),
    Input("mode", "value"),
    Input("mosaic_n", "value"),
    Input("display_space", "value"),
    prevent_initial_call=False,
)
def sync_axis(patient_id, selected_files, axis, mode, mosaic_n, display_space):
    if not patient_id or axis is None:
        raise PreventUpdate

    base_vol = get_base_volume(patient_id, display_space, selected_files)
    if base_vol is None:
        if not selected_files:
            raise PreventUpdate
        base_vol = load_patient_overlay(patient_id, selected_files[0], display_space)

    axis = int(axis)
    max_idx = base_vol.shape[axis] - 1
    note = f"T1 base: {base_vol.shape} | {AXIS_LABELS[axis]} | mode: {mode}"
    if mode == "mosaic":
        note += f" | mosaic_n: {mosaic_n}"
    return max_idx, max_idx // 2, note


@app.callback(
    Output("view", "figure"),
    Input("patient_dropdown", "value"),
    Input("nifti_dropdown", "value"),
    Input("overlay_settings", "data"),
    Input("mode", "value"),
    Input("axis", "value"),
    Input("idx", "value"),
    Input("mosaic_n", "value"),
    Input("gradient_settings", "data"),
    Input("display_space", "value")
)
def update_view(patient_id, selected_files, settings, mode, axis, idx, mosaic_n, gradient_settings, display_space):
    if not patient_id or mode is None or axis is None or idx is None or mosaic_n is None:
        raise PreventUpdate

    settings = settings or {}
    gradient_settings = gradient_settings or {}
    selected_files = selected_files or []
    axis = int(axis)
    idx = int(idx)
    mosaic_n = int(mosaic_n)

    base_vol = get_base_volume(patient_id, display_space, selected_files)

    print(f"[update_view] patient={patient_id} space={display_space} base_shape={base_vol.shape}")
    print(f"[update_view] t1_path={find_t1_file(patient_id)}")
    for f in selected_files:
        ov = load_patient_overlay(patient_id, f, display_space)
        print(f"[update_view] overlay {f} shape={ov.shape}")

    if base_vol is None:
        if not selected_files:
            return empty_figure("No T1 file found and no overlay selected")
        base_vol = load_patient_overlay(patient_id, selected_files[0], display_space)

    idx = max(0, min(idx, base_vol.shape[axis] - 1))
    overlay_vols = [load_patient_overlay(patient_id, f, display_space) for f in selected_files]
    per_overlay_settings = [
        settings.get(
            f,
            {
                "opacity": DEFAULT_OPACITY,
                "threshold": DEFAULT_THRESHOLD,
                "cluster_size": DEFAULT_CLUSTER_SIZE,
            },
        )
        for f in selected_files
    ]

    if mode == "single":
        rgb = blend_single_slice(
            base_vol=base_vol,
            overlay_vols=overlay_vols,
            axis=axis,
            idx=idx,
            per_overlay_settings=per_overlay_settings,
            gradient_settings=gradient_settings,
            selected_files=selected_files,
            display_space=display_space,
        )
        fig = px.imshow(rgb)
        fig.update_layout(
            template="plotly_white",
            margin=dict(l=10, r=10, t=50, b=10),
            title=f"{AXIS_LABELS[axis]} slice {idx}",
            coloraxis_showscale=False,
        )
        fig.update_xaxes(showticklabels=False)
        fig.update_yaxes(showticklabels=False)
        return fig

    mosaic_rgb, indices = make_mosaic_rgb(
        base_vol=base_vol,
        overlay_vols=overlay_vols,
        axis=axis,
        start=idx,
        step=1,
        n_slices=mosaic_n,
        pad=2,
        per_overlay_settings=per_overlay_settings,
        gradient_settings=gradient_settings,
        selected_files=selected_files,
        display_space=display_space
    )
    fig = px.imshow(mosaic_rgb)
    fig.update_layout(
        template="plotly_white",
        margin=dict(l=10, r=10, t=50, b=10),
        title=f"Mosaic | {AXIS_LABELS[axis]} | start={idx} | n={len(indices)}",
        coloraxis_showscale=False,
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig


@app.callback(
    Output("download_dicom", "data"),
    Input("export_dicom_button", "n_clicks"),
    State("patient_dropdown", "value"),
    State("nifti_dropdown", "value"),
    State("overlay_settings", "data"),
    State("mode", "value"),
    State("axis", "value"),
    State("idx", "value"),
    State("mosaic_n", "value"),
    State("gradient_settings", "data"),
    State("display_space", "value"),
    prevent_initial_call=True,
)
def export_current_view_as_dicom(n_clicks, patient_id, selected_files, settings, mode, axis, idx, mosaic_n, gradient_settings, display_space):
    if not n_clicks:
        raise PreventUpdate

    rgb_u8, stem = render_current_rgb(
        patient_id=patient_id,
        selected_files=selected_files or [],
        settings=settings or {},
        mode=mode,
        axis=axis,
        idx=idx,
        mosaic_n=mosaic_n,
        gradient_settings=gradient_settings or {},
        display_space=display_space,
    )

    from PIL import Image

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dicom_filename = f"{stem}_{timestamp}.dcm"
    png_filename = f"{stem}_{timestamp}.png"

    tmp_path = os.path.join(BASE_DIR, dicom_filename)
    png_path = os.path.join(BASE_DIR, png_filename)

    Image.fromarray(rgb_u8).save(png_path)

    save_rgb_secondary_capture(
        rgb_u8=rgb_u8,
        out_path=tmp_path,
        patient_id=patient_id,
        series_description=f"Dash overlay export ({mode})",
    )

    print("Saved PNG to:", png_path)
    print("Saved DICOM to:", tmp_path)

    return dcc.send_file(tmp_path, filename=dicom_filename)

@app.callback(
    Output("overall_patient_rating_dropdown", "value"),
    Input("patient_dropdown", "value"),
    State("overall_patient_ratings", "data"),
)
def load_patient_rating(patient_id, ratings):

    if not patient_id:
        raise PreventUpdate

    ratings = ratings or {}

    return ratings.get(patient_id)

@app.callback(
    Output("overall_patient_ratings", "data"),
    Input("overall_patient_rating_dropdown", "value"),
    State("patient_dropdown", "value"),
    State("overall_patient_ratings", "data"),
    prevent_initial_call=True,
)
def save_overall_patient_rating(rating, patient_id, current):

    if patient_id is None:
        raise PreventUpdate

    data = dict(current or {})
    data[patient_id] = rating

    return data

@app.callback(
    Output("download_volume", "data"),
    Input("export_volume_button", "n_clicks"),
    State("patient_dropdown", "value"),
    State("nifti_dropdown", "value"),
    State("overlay_settings", "data"),
    State("gradient_settings", "data"),
    State("display_space", "value"),
    prevent_initial_call=True,
)
def export_full_volume(
    n_clicks,
    patient_id,
    selected_files,
    settings,
    gradient_settings,
    display_space,
):
    if not n_clicks:
        raise PreventUpdate

    rgb_volume = render_full_rgb_volume(
        patient_id=patient_id,
        selected_files=selected_files or [],
        settings=settings or {},
        gradient_settings=gradient_settings or {},
        display_space=display_space,
    )

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    export_dir = os.path.join(
        BASE_DIR,
        f"{patient_id}_volume_export_{timestamp}"
    )

    save_rgb_volume_as_dicom_series(
        rgb_volume,
        export_dir,
        patient_id
    )

    import shutil

    zip_path = shutil.make_archive(
        export_dir,
        "zip",
        export_dir
    )

    return dcc.send_file(zip_path)

# =========================
# 6) RUN
# =========================
if __name__ == "__main__":
    app.run(debug=False, host="0.0.0.0", port=8051, use_reloader=False)

SELECTED FILES: []
[update_view] patient=hc01jbw space=subject base_shape=(176, 224, 224)
[update_view] t1_path=/Users/forrestlin/Downloads/example_output_mni/hc01jbw/vout/sub-hc01jbw/ses-1/anat/sub-hc01jbw_ses-1_desc-preproc_T1w.nii.gz
SELECTED FILES: []
[update_view] patient=hc01jbw space=subject base_shape=(176, 224, 224)
[update_view] t1_path=/Users/forrestlin/Downloads/example_output_mni/hc01jbw/vout/sub-hc01jbw/ses-1/anat/sub-hc01jbw_ses-1_desc-preproc_T1w.nii.gz
SELECTED FILES: []
[update_view] patient=hc02jbw space=subject base_shape=(176, 224, 224)
[update_view] t1_path=/Users/forrestlin/Downloads/example_output_mni/hc02jbw/vout/sub-hc02jbw/ses-1/anat/sub-hc02jbw_ses-1_desc-preproc_T1w.nii.gz
SELECTED FILES: []
[update_view] patient=hc03jbw space=subject base_shape=(176, 224, 224)
[update_view] t1_path=/Users/forrestlin/Downloads/example_output_mni/hc03jbw/vout/sub-hc03jbw/ses-1/anat/sub-hc03jbw_ses-1_desc-preproc_T1w.nii.gz
SELECTED FILES: ['sub-hc03jbw_ses-1_task-rest_acq-mu